# News clustering

Objective: The goal is to partition news articles from a specific timeframe (a 7-day sliding window) into semantically correlated groups, where each cluster ideally represents a real-world financial event.
- Algorithm: Use Agglomerative Clustering (hierarchical). This is a bottom-up approach where each article begins as its own cluster before being successively merged with others.
- Distance Metric: Use Cosine Distance, which is the most suitable metric for comparing high-dimensional text vectors.
- Linkage Criterion: Use Average Linkage, which minimizes the average distance between all observations of pairs of clusters.
- Determining $k$ (Number of Clusters): Since the number of events fluctuates daily, the optimal $k$ is found using the Silhouette Maximization method.
    - Iterate the algorithm for $k$ values ranging from 2 to 10.
    - Calculate the average Silhouette Coefficient for each $k$.
    - Select the value of $k$ that yields the highest score.
- Centroid Calculation: Once clusters are formed, compute a centroid for each group using the median of the cluster embeddings, as it is more robust to noise than the mean.

In [1]:
import pandas as pd
import numpy as np
import os
import sys

sys.path.append(os.path.abspath(os.path.join("..")))
from src.news_clustering import *

### Data preparation

In [2]:
# --- 1. FONCTION DE PARSING DES EMBEDDINGS ---
def parse_embedding_string(emb_str):
    """
    Convertit la chaîne de caractères du CSV (ex: "[ 0.12 \n -0.34 ...]")
    en un vrai tableau numpy utilisable pour le clustering.
    """
    clean_str = str(emb_str).replace("[", "").replace("]", "").replace("\n", " ")
    return np.fromstring(clean_str, sep=" ")

In [3]:
# --- 2. CHARGEMENT DES DONNÉES ---
# Word2Vec
df_w2v = pd.read_csv("../data/for_models/news_features_w2v.csv")
X_w2v = np.stack(df_w2v["embedding"].apply(parse_embedding_string).values)

# DistilRoBERTa (BPE)
df_bpe = pd.read_csv("../data/for_models/news_features_bpe.csv")
X_bpe = np.stack(df_bpe["embedding"].apply(parse_embedding_string).values)

print(f"Format Word2Vec : {X_w2v.shape} (Attendus: Nx300)")
print(f"Format DistilRoBERTa : {X_bpe.shape} (Attendus: Nx768)")

Format Word2Vec : (692, 300) (Attendus: Nx300)
Format DistilRoBERTa : (736, 768) (Attendus: Nx768)


### Choice of k clusters

In [4]:
# Exemple : Choice of clusters for the week before the IRAN vs US/Israel event
START_DATE = "2026-02-26"
END_DATE = "2026-03-03"

In [5]:
print(f"\n--- ANALYSE DE LA PÉRIODE : {START_DATE} au {END_DATE} ---")

# HAC Evaluation for both embeddings models
print("\nÉvaluation HAC sur Word2Vec...")
results_w2v = run_hac_evaluation_period(
    X_full=X_w2v,
    df_features=df_w2v,
    start_date=START_DATE,
    end_date=END_DATE,
    k_range=range(2, 10),
)
plot_hac_evaluation(
    results_w2v, title=f"Word2Vec K-Evaluation ({START_DATE} to {END_DATE})"
)

print("\nÉvaluation HAC sur DistilRoBERTa (BPE)...")
results_bpe = run_hac_evaluation_period(
    X_full=X_bpe,
    df_features=df_bpe,
    start_date=START_DATE,
    end_date=END_DATE,
    k_range=range(2, 10),
)
plot_hac_evaluation(
    results_bpe, title=f"DistilRoBERTa K-Evaluation ({START_DATE} to {END_DATE})"
);


--- ANALYSE DE LA PÉRIODE : 2026-02-26 au 2026-03-03 ---

Évaluation HAC sur Word2Vec...



Évaluation HAC sur DistilRoBERTa (BPE)...


So, the adequate number of clusters for the week before this event seems to be 2 or 3.

In [6]:
# ÉTAPE C : L'utilisateur regarde les graphiques et choisit le meilleur K pour CHAQUE modèle
best_k_w2v = 2  # À lire sur le graphique Word2Vec
best_k_bpe = 2  # À lire sur le graphique DistilRoBERTa

# --- 4. VISUALISATION DES CLUSTERS (Exemple sur une période de crise) ---
print(f"\nVisualisation 2D (Word2Vec) avec K={best_k_w2v}...")
fig_tsne_w2v = visualize_hac_tsne_range(
    X_full=X_w2v,
    df_features=df_w2v,
    start_date=START_DATE,
    end_date=END_DATE,
    k=best_k_w2v,
)
if fig_tsne_w2v is not None:
    fig_tsne_w2v.show()

print(f"\nVisualisation 2D (DistilRoBERTa) avec K={best_k_bpe}...")
fig_tsne_bpe = visualize_hac_tsne_range(
    X_full=X_bpe,
    df_features=df_bpe,
    start_date=START_DATE,
    end_date=END_DATE,
    k=best_k_bpe,
)
if fig_tsne_bpe is not None:
    fig_tsne_bpe.show()


Visualisation 2D (Word2Vec) avec K=2...



Visualisation 2D (DistilRoBERTa) avec K=2...


In [7]:
# ÉTAPE E : Dendrogrammes finaux pour les DEUX modèles
print("\nGénération du Dendrogramme (Word2Vec)...")
Z_w2v, labels_w2v, _ = compute_stable_hac_linkage(
    X_full=X_w2v,
    df_features=df_w2v,
    start_date=START_DATE,
    end_date=END_DATE,
    k=best_k_w2v,
)
fig_dendro_w2v = plot_hac_dendrogram_plotly(Z_w2v, labels_w2v, START_DATE, END_DATE)
if fig_dendro_w2v is not None:
    fig_dendro_w2v.show()

print("\nGénération du Dendrogramme (DistilRoBERTa)...")
Z_bpe, labels_bpe, _ = compute_stable_hac_linkage(
    X_full=X_bpe,
    df_features=df_bpe,
    start_date=START_DATE,
    end_date=END_DATE,
    k=best_k_bpe,
)
fig_dendro_bpe = plot_hac_dendrogram_plotly(Z_bpe, labels_bpe, START_DATE, END_DATE)
if fig_dendro_bpe is not None:
    fig_dendro_bpe.show()


Génération du Dendrogramme (Word2Vec)...



Génération du Dendrogramme (DistilRoBERTa)...


# Outliers removal

Objective : Denoising the news clusters by removing outliers usig the Double Filtering Logic.
The paper assumes that clustering can fail in two distinct ways:
- Ambiguity (Silhouette): The article is "caught between two fires," being almost as close to a neighboring cluster as it is to its own.I
- solation (Distance to Centroid): The article is in the correct cluster, but it is located very far from the center (indicating a topic that is too specific or off-topic).

Steps :
- Retrieve Initial Centroids: Computed after the HAC process.
- Compute Two Metrics for each article $i$:
    - Its individual Silhouette score ($s_i$).
    - Its Cosine Similarity with the centroid of its assigned cluster.
- Define Cutoff Thresholds by Percentile (e.g., 20th percentile):
    -  Identify the 30% of articles with the worst (lowest) Silhouette scores.
    -  Identify the 30% of articles furthest from their centroids (lowest similarity).
- Suppression: Remove an article if it fails either of the two tests (Logical "OR").
- Recalculate Final Centroids: Compute the final event signatures based only on the remaining "clean" articles.

In [8]:
# ==========================================
# ÉTAPE F : EXTRACTION DES CLUSTERS STABLES
# ==========================================
print("\n--- EXTRACTION ET NETTOYAGE DES OUTLIERS ---")

# Word2Vec
X_stable_w2v, df_stable_w2v, labels_w2v = get_stable_clusters(
    X_full=X_w2v,
    df_features=df_w2v,
    start_date=START_DATE,
    end_date=END_DATE,
    k=best_k_w2v,
)

# DistilRoBERTa
X_stable_bpe, df_stable_bpe, labels_bpe = get_stable_clusters(
    X_full=X_bpe,
    df_features=df_bpe,
    start_date=START_DATE,
    end_date=END_DATE,
    k=best_k_bpe,
)

# ==========================================
# ÉTAPE G : ADVANCED OUTLIER REMOVAL (Double Filtre)
# ==========================================
PERCENTILE_CUTOFF = 20  # On élimine les 20% les plus ambigus/isolés

if X_stable_w2v is not None:
    mask_keep_w2v = remove_news_outliers_advanced(
        X_stable_w2v, labels_w2v, percentile_threshold=PERCENTILE_CUTOFF
    )

    X_clean_w2v = X_stable_w2v[mask_keep_w2v]
    df_clean_w2v = df_stable_w2v[mask_keep_w2v].copy()

    print(
        f"[Word2Vec] Articles initiaux : {len(X_stable_w2v)} | Après nettoyage : {len(X_clean_w2v)}"
    )

    # Recalculer les centroïdes (Signatures d'Événements) sur les clusters PROPRES
    final_centroids_w2v = calculate_event_centroids(
        X_clean_w2v, df_clean_w2v["Cluster"].values
    )

if X_stable_bpe is not None:
    mask_keep_bpe = remove_news_outliers_advanced(
        X_stable_bpe, labels_bpe, percentile_threshold=PERCENTILE_CUTOFF
    )

    X_clean_bpe = X_stable_bpe[mask_keep_bpe]
    df_clean_bpe = df_stable_bpe[mask_keep_bpe].copy()

    print(
        f"[DistilRoBERTa] Articles initiaux : {len(X_stable_bpe)} | Après nettoyage : {len(X_clean_bpe)}"
    )

    # Recalculer les centroïdes (Signatures d'Événements) sur les clusters PROPRES
    final_centroids_bpe = calculate_event_centroids(
        X_clean_bpe, df_clean_bpe["Cluster"].values
    )

# ==========================================
# ÉTAPE H : SAUVEGARDE POUR L'ASSIGNATION DES TWEETS
# ==========================================
print("\n--- SAUVEGARDE DES FICHIERS PURIFIÉS ---")

# 1. Sauvegarde pour Word2Vec
if X_stable_w2v is not None:
    path_w2v = f"../data/for_models/clean_news_w2v_{START_DATE}_to_{END_DATE}.csv"
    df_clean_w2v.to_csv(path_w2v, index=False)
    print(f"[Word2Vec] Fichier propre sauvegardé : {path_w2v}")

# 2. Sauvegarde pour DistilRoBERTa
if X_stable_bpe is not None:
    path_bpe = f"../data/for_models/clean_news_bpe_{START_DATE}_to_{END_DATE}.csv"
    df_clean_bpe.to_csv(path_bpe, index=False)
    print(f"[DistilRoBERTa] Fichier propre sauvegardé : {path_bpe}")


--- EXTRACTION ET NETTOYAGE DES OUTLIERS ---
[Word2Vec] Articles initiaux : 54 | Après nettoyage : 37
[DistilRoBERTa] Articles initiaux : 54 | Après nettoyage : 36

--- SAUVEGARDE DES FICHIERS PURIFIÉS ---
[Word2Vec] Fichier propre sauvegardé : ../data/for_models/clean_news_w2v_2026-02-26_to_2026-03-03.csv
[DistilRoBERTa] Fichier propre sauvegardé : ../data/for_models/clean_news_bpe_2026-02-26_to_2026-03-03.csv
